In [2]:
import warnings
warnings.filterwarnings("ignore")

from rfdetr import RFDETRLarge
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import os
import optuna

In [3]:
def objective(trial):
    lr = trial.suggest_float("lr", 3e-5, 3e-4, log=True)
    lr_encoder = trial.suggest_float("lr_encoder", 3e-5, 3e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    model = RFDETRLarge(
        resolution=336,
        gradient_checkpointing=True
    )

    output_dir = f"runs/optuna/encoder/trial_{trial.number}"

    model.train(
        dataset_dir=r"data/rfdetr",
        epochs=40,
        batch_size=4,
        grad_accum_steps=8,

        lr=lr,
        lr_encoder=lr_encoder,
        weight_decay=weight_decay,
        lr_scheduler="cosine",

        use_ema=True,

        early_stopping=True,
        early_stopping_patience=5,
        early_stopping_min_delta=1e-3,
        skip_best_epochs=5,

        eval_interval=1,
        seed=42,

        output_dir=output_dir
    )

    metrics = model.evaluate(
        dataset_dir=r"data/rfdetr",
        split="val"
    )

    map50 = metrics["val/mAP_50"]

    return map50

In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(sampler= sampler, direction="maximize")
study.optimize(objective, n_trials=10)

Val (ema) (Epoch 1/40) — Overall Metrics            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.1887 │ 0.3040 │ 0.1931 │ 0.6215 │ 0.2732 │ 0.2378 │ 0.4077 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [5]:
best_trial = study.best_trial
print(best_trial.number)
print(best_trial.value)
print(best_trial.params)

1
0.7225565910339355
{'lr': 6.964583207205953e-05, 'lr_encoder': 0.00024250150081520832, 'weight_decay': 1.54603838402487e-05}


In [ ]:
# 1(best) 4 5 7 8 high
# 2 3 6 low
NUM_CLASSES = 8

best_checkpoint = Path(
    f"runs/optuna/encoder/trial_{best_trial.number}"
) 
# best_checkpoint = Path(
#     f"runs/optuna/encoder/trial_6"
# ) 

model = RFDETRLarge(
    pretrain_weights= best_checkpoint / "checkpoint_best_total.pth"
)

image_dir = Path(r"data/raw/test/test")

rows = []
row_id = 0

for image_path in tqdm(list(image_dir.glob("*")), desc="Predicting"):

    detections = model.predict(
        str(image_path),
        threshold=0.01
    )

    for box, conf, cls_id in zip(
        detections.xyxy,
        detections.confidence,
        detections.class_id
    ):
        cls_id = int(cls_id)

        # Skip background / invalid class
        if not 0 <= cls_id < NUM_CLASSES:
            continue

        x1, y1, x2, y2 = map(float, box)

        # Skip degenerate boxes
        if x2 <= x1 or y2 <= y1:
            continue

        rows.append({
            "id": row_id,
            "image_id": image_path.name,
            "class_id": cls_id,
            "confidence": float(conf),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

        row_id += 1

df_pred = pd.DataFrame(
    rows,
    columns=[
        "id",
        "image_id",
        "class_id",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2",
    ]
)

# Final validation
assert df_pred["class_id"].between(0, 7).all()
assert (df_pred["x2"] > df_pred["x1"]).all()
assert (df_pred["y2"] > df_pred["y1"]).all()

df_pred.to_csv(
    best_checkpoint / "submission.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Predictions:", len(df_pred))

[2026-09-29 16:44:57] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-29 16:44:57] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-29 16:44:58] [WARNING] rf-detr - Checkpoint has 8 classes but model is configured for 90. Using checkpoint class count (8). Pass num_classes=8 to suppress this warning.
Predicting: 100%|██████████| 1013/1013 [00:52<00:00, 19.38it/s]


Predictions: 283378
